# Aula 02 — Detecção de nuvem e sombra com OmniCloudMask

Nessa aula iremos utilizar o **OmniCloudMask** para detectar nuvens e
sombras na cena de 2 m que geramos na aula anterior. O resultado é uma
**máscara georreferenciada** que indica quais pixels estão cobertos por
nuvem ou sombra.

A cena de 2 m que saiu da Aula 01 tem nuvem em parte da área. Aqui a
gente descobre **exatamente onde**, e grava isso como uma máscara
georreferenciada — que é o que a Aula 03 vai usar para saber quais
pixels precisam ser substituídos.

O **OmniCloudMask** (OCM) (Wright et al., 2025) é uma rede neural
convolucional treinada em cenas de vários sensores para separar quatro
classes:

| valor | classe | o que é |
|------------------------|------------------------|------------------------|
| 0 | `clear` | céu limpo — pixel utilizável |
| 1 | `thick_cloud` | nuvem espessa, opaca |
| 2 | `thin_cloud` | nuvem fina, cirros — deixa passar parte do sinal do solo |
| 3 | `cloud_shadow` | sombra projetada pela nuvem |

Ele usa só três bandas — **vermelho, verde e infravermelho próximo** — e
é **agnóstico ao sensor**: não foi treinado com CBERS-4A e mesmo assim
funciona, porque aprendeu a assinatura espectral e a textura da nuvem,
não as particularidades de um satélite. Isso só é possível porque o OCM
foi treinado com um dataset enorme, com alta diversidade de sensores,
resoluções espectrais e com imagens de todo o mundo. (Para mais
detalhes, consultar Wright et al. (2025).)

> **Detalhe importante:** apesar de ser extremamente robusto, ainda há
> erros. Algumas sombras de nuvens podem passar despercebidas e algumas
> nuvens finas podem ser confundidas com alvos claros.

## Preparando o ambiente

**No pixi local não faz nada**: `pixi install` já trouxe tudo.

**No Colab**, a primeira célula instala o conda na sessão e **reinicia o
ambiente**. Ela só é necessária se você **não** fez as aulas anteriores
nesta mesma sessão — o atalho que refaz a Aula 01 precisa do 6S para a
correção BOA. Se você já tem o `pansharpening_*.tif`, pode pular direto
para a segunda célula.

In [1]:
# --- Colab, parte 1 de 2: instala o conda na sessão (REINICIA o ambiente) ---
import sys

NO_COLAB = "google.colab" in sys.modules
print("Ambiente:", "Google Colab" if NO_COLAB else "local (pixi/Jupyter)")

if NO_COLAB:
    !pip install -q condacolab
    import condacolab
    condacolab.install()   # o ambiente reinicia aqui — é esperado

Ambiente: local (pixi/Jupyter)

In [2]:
# --- Colab, parte 2 de 2: rode DEPOIS do reinício ---
import sys

NO_COLAB = "google.colab" in sys.modules
if NO_COLAB:
    !mamba install -q -y -c conda-forge rasterio pyproj py6s sixs pystac-client requests
    !pip install -q omnicloudmask
    print("pronto")

In [3]:
# --- Módulo de apoio e configuração ---
from pathlib import Path

REPO = "https://github.com/danfioramonte/geoprocessamento-cbers4a"

if not Path("aulas_cbers.py").exists():
    !git clone -q --depth 1 {REPO} /tmp/geo_repo
    !cp /tmp/geo_repo/colab/aulas_cbers.py .
    !cp -r /tmp/geo_repo/colab/config .

import aulas_cbers as ac
ac.configurar_gdal()

import json
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.colors import BoundaryNorm, ListedColormap
from rasterio import Affine
from rasterio.enums import Resampling
from rasterio.warp import reproject
from rasterio.windows import Window

## 0. Configuração

`DETECT_RES` é a decisão mais importante desta aula, e a próxima seção
explica por quê.

In [4]:
CENA_ID = ac.CENA_PADRAO
NIVEL = "boa"                 # tem que casar com o que você gerou na Aula 01
DETECT_RES = 10.0             # resolução (m) em que a detecção acontece
BANDA_RED, BANDA_GREEN, BANDA_NIR = 1, 2, 4     # ordem RGBN do produto da Aula 01
PATCH, OVERLAP = 1000, 300    # janelamento interno do OCM
NODATA = ac.NODATA
CLASSES = ac.CLASSES_OCM
NODATA_MASCARA = 255          # 255 = fora do dado, nas máscaras de saída

P = ac.pastas()
DATA = ac.data_da_cena(CENA_ID)

entrada = P["interim"] / f"pansharpening_{DATA}_{NIVEL}.tif"
if not entrada.exists():
    # atalho para quem começou o curso por esta aula: refaz Aula 00 + Aula 01
    r = ac.preparar_aula_02(CENA_ID, NIVEL)
    NIVEL, entrada = r["nivel"], Path(r["pansharpening"])

with rasterio.open(entrada) as src:
    print(f"entrada : {entrada.name}")
    print(f"          {src.width} x {src.height} px @ {src.res[0]:g} m | {src.dtypes[0]}")
    print(f"bandas  : {src.descriptions}")
    print(f"nodata  : {src.nodata} | CRS: {src.crs}")

entrada : pansharpening_20260301_boa.tif
          6456 x 6968 px @ 2 m | int16
bandas  : ('red', 'green', 'blue', 'nir')
nodata  : -9999.0 | CRS: EPSG:32723

## 1. Por que detectar em 10 m, e não nos 2 m nativos

Três motivos:

1.  **O OCM foi validado na faixa de 10 a 50 m.** Fora dela o desempenho
    cai — a rede aprendeu a nuvem numa determinada escala de textura, e
    em 2 m ela vê estrutura demais.
2.  **Nuvem é um objeto grande.** A borda de uma nuvem não é nítida em
    escala nenhuma: existe uma faixa de transição de dezenas de metros.
    Detectar em 2 m não dá uma borda mais correta, dá uma borda mais
    ruidosa.
3.  **Custo.** São 16 vezes menos pixels para a rede processar.

A máscara sai em 10 m e depois volta para a grade nativa de 2 m por
**vizinho mais próximo** — o único reamostrador aceitável para dado
categórico.

In [5]:
with rasterio.open(entrada) as src:
    nativo = {"transform": src.transform, "crs": src.crs,
              "height": src.height, "width": src.width, "res": src.res[0]}
    escala = src.res[0] / DETECT_RES
    h = max(1, int(round(src.height * escala)))
    w = max(1, int(round(src.width * escala)))

    # bandas na ordem exigida pelo OCM: Red, Green, NIR
    det = src.read([BANDA_RED, BANDA_GREEN, BANDA_NIR], out_shape=(3, h, w),
                   resampling=Resampling.average).astype("float32")

    # Validade. Duas coisas acontecem aqui, e vale saber a diferença:
    #  - a média do GDAL IGNORA os pixels de nodata, então um bloco de 10 m que
    #    cai meio em cima da borda sai com a média só do que existe, sem
    #    contaminação. Bom: não perdemos a borda.
    #  - o que a média não diz é ONDE não sobrou nada. Para isso lemos a MÁSCARA
    #    do arquivo (255 onde há dado, 0 onde não há), reduzida do mesmo jeito:
    #    ela zera só nos blocos inteiramente sem dado.
    # O teste extra em det é cinto e suspensório, caso alguma versão do GDAL
    # espalhe o -9999 pela média em vez de ignorá-lo.
    mascara = src.read_masks(1, out_shape=(h, w), resampling=Resampling.average)
    t_det = src.transform * Affine.scale(src.width / w, src.height / h)

valido = (mascara >= 255) & np.all(det > -5000, axis=0)
det[:, ~valido] = 0.0     # o OCM ignora estes pixels (no_data_value=0)

print(f"grade de detecção : {w} x {h} px @ {DETECT_RES:g} m")
print(f"pixels válidos    : {valido.sum():,} ({valido.mean():.2%})")

grade de detecção : 1291 x 1394 px @ 10 m
pixels válidos    : 1,799,637 (100.00%)

## 2. Rodando o OmniCloudMask

Na primeira execução ele baixa os pesos do modelo via Hugging Face (HF).

O OCM trabalha em blocos de `PATCH` pixels com `OVERLAP` de sobreposição
e faz a média nas emendas.

``` python
import torch
from omnicloudmask import predict_from_array

print("dispositivo:", "GPU (cuda)" if torch.cuda.is_available() else "CPU")

mask = predict_from_array(
    det,
    patch_size=PATCH,
    patch_overlap=OVERLAP,
    no_data_value=0,
    apply_no_data_mask=True,
    model_download_source="hugging_face",
)
mask = np.squeeze(np.asarray(mask)).astype("uint8")
mask = np.where(valido, mask, NODATA_MASCARA).astype("uint8")

total = int(valido.sum())
print("\n--- composição da cena (pixels válidos, grade de detecção) ---")
for c, nome in CLASSES.items():
    n = int(np.count_nonzero((mask == c) & valido))
    print(f"  {c} {nome:<13}: {n:>10,}  ({100 * n / max(total, 1):5.1f} %)")
contaminado = int(np.count_nonzero(np.isin(mask, [1, 2, 3]) & valido))
print(f"  -> nuvem + sombra: {contaminado:>10,}  ({100 * contaminado / max(total, 1):5.1f} %)")
print(f"  -> utilizável    : {total - contaminado:>10,}  "
      f"({100 * (total - contaminado) / max(total, 1):5.1f} %)")
```

    C:\geo\geoprocessamento-cbers4a\.pixi\envs\default\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
      from .autonotebook import tqdm as notebook_tqdm

    dispositivo: CPU

    --- composição da cena (pixels válidos, grade de detecção) ---
      0 clear        :  1,597,022  ( 88.7 %)
      1 thick_cloud  :    123,299  (  6.9 %)
      2 thin_cloud   :          0  (  0.0 %)
      3 cloud_shadow :     79,316  (  4.4 %)
      -> nuvem + sombra:    202,615  ( 11.3 %)
      -> utilizável    :  1,597,022  ( 88.7 %)

Tabela 1

## 3. Verificando antes de continuar

Nenhuma máscara de nuvem é perfeita, e a hora de descobrir os erros é
agora. O notebook mostra a falsa cor e a máscara lado a lado, mais um
zoom na maior mancha de nuvem.

O que conferir:

- **sombra deslocada**: a sombra fica do lado oposto ao Sol em relação à
  nuvem. Se a classe 3 aparece do lado errado, alguma banda está trocada
  na entrada;
- **nuvem fina confundida com alvo claro**: telhado metálico, areia e
  pátio de concreto às vezes entram como classe 2. Numa área urbana isso
  acontece mais;
- **borda**: uma faixa de classe 2 em volta da nuvem espessa é
  **esperada e desejável** — é a transição real, e é melhor que ela
  entre na máscara do que fique de fora;
- **água**: corpo d’água escuro às vezes vira sombra. Se a sua AOI tem
  represa, olhe com atenção.

In [7]:
def stretch(b, m):
    v = b[m]
    if v.size == 0:
        return np.zeros_like(b)
    lo, hi = np.percentile(v, [2, 98])
    return np.clip((b - lo) / (hi - lo + 1e-6), 0, 1)


cores = ListedColormap(["#2c7fb8", "#f0f0f0", "#fdae6b", "#54278f"])
norma = BoundaryNorm([-.5, .5, 1.5, 2.5, 3.5], cores.N)

r, g, nir = det[0], det[1], det[2]
falsa_cor = np.dstack([stretch(nir, valido), stretch(r, valido), stretch(g, valido)])
falsa_cor[~valido] = 0

fig, ax = plt.subplots(1, 2, figsize=(15, 7.5))
ax[0].imshow(falsa_cor)
ax[0].set_title("Falsa cor NIR/R/G — nuvem em branco, vegetação em vermelho")
im = ax[1].imshow(np.where(valido, mask, np.nan), cmap=cores, norm=norma)
ax[1].set_title("Máscara OmniCloudMask")
cb = fig.colorbar(im, ax=ax[1], ticks=[0, 1, 2, 3], fraction=0.046, pad=0.04)
cb.ax.set_yticklabels(["clear", "thick", "thin", "shadow"])
for a in ax:
    a.axis("off")
plt.tight_layout(); plt.show()

In [8]:
# zoom na maior mancha de nuvem, para conferir a borda de perto
nuvem = np.isin(mask, [1, 2])
if nuvem.any():
    ys, xs = np.nonzero(nuvem)
    cy, cx = int(ys.mean()), int(xs.mean())
    lado = min(160, h // 2, w // 2)
    y0 = max(0, min(h - 2 * lado, cy - lado))
    x0 = max(0, min(w - 2 * lado, cx - lado))
    fatia = (slice(y0, y0 + 2 * lado), slice(x0, x0 + 2 * lado))

    fig, ax = plt.subplots(1, 2, figsize=(13, 6.6))
    ax[0].imshow(falsa_cor[fatia])
    ax[0].set_title("zoom — falsa cor")
    ax[1].imshow(falsa_cor[fatia])
    ax[1].imshow(np.where(np.isin(mask[fatia], [1, 2, 3]), mask[fatia], np.nan),
                 cmap=cores, norm=norma, alpha=0.55)
    ax[1].set_title("zoom — máscara sobreposta")
    for a in ax:
        a.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("Nenhum pixel de nuvem detectado nesta cena.")
    print("Para as Aulas 02 e 03 fazerem sentido, escolha na Aula 00 uma cena com nuvem "
          "sobre a AOI (a coluna 'nuvens_est_%' da tabela de diagnóstico ajuda).")

## 4. Gravando as máscaras

Três arquivos, com papéis diferentes:

- **`_cloudmask_10m.tif`** — a máscara na grade em que a rede realmente
  decidiu. É a que você usa para auditar o resultado;
- **`_cloudmask_native.tif`** — a mesma, reamostrada para os 2 m da
  cena. É a que a Aula 03 consome, e ela **precisa** estar exatamente na
  grade do CBERS;
- **`_clearmask_native.tif`** — versão binária (1 = utilizável).
  Conveniente para multiplicar por outros rasters.

In [9]:
def gravar_mascara(caminho, arr, transform):
    perfil = dict(driver="GTiff", dtype="uint8", count=1,
                  height=arr.shape[0], width=arr.shape[1], transform=transform,
                  crs=nativo["crs"], nodata=NODATA_MASCARA, compress="deflate",
                  tiled=True, blockxsize=256, blockysize=256, BIGTIFF="IF_SAFER")
    with rasterio.open(caminho, "w", **perfil) as dst:
        dst.write(arr.astype("uint8"), 1)
        dst.set_band_description(1, "ocm_classe")
        dst.update_tags(CLASSES="0=clear,1=thick_cloud,2=thin_cloud,3=cloud_shadow,255=nodata")
    print(f"Escrito: {caminho.name}")
    return caminho


stem = entrada.stem
cam_det = gravar_mascara(P["interim"] / f"{stem}_cloudmask_{int(DETECT_RES)}m.tif",
                         mask, t_det)

nativa = np.full((nativo["height"], nativo["width"]), NODATA_MASCARA, dtype="uint8")
reproject(source=mask, destination=nativa,
          src_transform=t_det, src_crs=nativo["crs"],
          dst_transform=nativo["transform"], dst_crs=nativo["crs"],
          resampling=Resampling.nearest)          # categórico: só vizinho mais próximo
cam_nat = gravar_mascara(P["interim"] / f"{stem}_cloudmask_native.tif",
                         nativa, nativo["transform"])

clear = (nativa == 0).astype("uint8")
clear[nativa == NODATA_MASCARA] = NODATA_MASCARA
cam_clear = gravar_mascara(P["interim"] / f"{stem}_clearmask_native.tif",
                           clear, nativo["transform"])

Escrito: pansharpening_20260301_boa_cloudmask_10m.tif
Escrito: pansharpening_20260301_boa_cloudmask_native.tif
Escrito: pansharpening_20260301_boa_clearmask_native.tif

## 5. Aplicando a máscara na imagem

Agora gravamos uma cópia da cena em que nuvem e sombra viram **nodata**.

Um pixel zerado entra em qualquer estatística puxando a média para baixo
sem avisar. Como nodata, ele é ignorado pelo rasterio, pelo seu software
de GIS e pelo AROSICS, que não vai tentar achar ponto de amarração
dentro da área removida.

In [10]:
mascarada = P["interim"] / f"{stem}_masked.tif"
nublado = np.isin(nativa, [1, 2, 3])

with rasterio.open(entrada) as src:
    perfil = src.profile.copy()
    nd = src.nodata if src.nodata is not None else NODATA
    for k in ("blockxsize", "blockysize", "tiled", "interleave"):
        perfil.pop(k, None)
    perfil.update(nodata=nd, compress="deflate", predictor=2, tiled=True,
                  blockxsize=512, blockysize=512, BIGTIFF="IF_SAFER",
                  interleave="band")
    descr = src.descriptions

    with rasterio.open(mascarada, "w", **perfil) as dst:
        for b in range(1, src.count + 1):
            for r0 in range(0, src.height, 2048):
                nr = min(2048, src.height - r0)
                jan = Window(0, r0, src.width, nr)
                dados = src.read(b, window=jan)
                dados[nublado[r0:r0 + nr, :]] = nd
                dst.write(dados, b, window=jan)
            dst.set_band_description(b, descr[b - 1] or ac.NOMES_SAIDA[b - 1])
        dst.update_tags(MASCARA=str(cam_nat.name), CLASSES_MASCARADAS="1,2,3")

print(f"Escrito: {mascarada.name}  ({mascarada.stat().st_size / 1024**2:.1f} MB)")

Escrito: pansharpening_20260301_boa_masked.tif  (214.5 MB)

In [11]:
# antes x depois, em resolução reduzida
with rasterio.open(mascarada) as src:
    fator = max(1, src.width // 800)
    forma = (src.height // fator, src.width // fator)
    dep = src.read((1, 2, 3), out_shape=(3,) + forma,
                   resampling=Resampling.average).astype("float32")
with rasterio.open(entrada) as src:
    ant = src.read((1, 2, 3), out_shape=(3,) + forma,
                   resampling=Resampling.average).astype("float32")

for arr in (ant, dep):
    arr[arr <= -9000] = np.nan

fig, ax = plt.subplots(1, 2, figsize=(15, 7.5))
ax[0].imshow(np.dstack([ac.stretch(b) for b in ant]))
ax[0].set_title("Aula 01 — cena fusionada")
ax[1].imshow(np.dstack([ac.stretch(b) for b in dep]))
ax[1].set_title("Aula 02 — nuvem e sombra removidas (as lacunas da Aula 03)")
for a in ax:
    a.axis("off")
plt.tight_layout(); plt.show()

## 6. Vale a pena preencher?

Se a nuvem cobre 1% da AOI, muitas vezes compensa descartar esses pixels
e seguir. Se cobre 25%, descartar significa perder um quarto da área — e
aí a substituição com Sentinel-2 super-resolvido paga o trabalho.

Não existe limiar universal. O que existe é a informação — fração
contaminada, área em hectares, composição por classe — e a sua decisão
**registrada**. O notebook grava tudo isso no `.params.json` ao lado da
máscara.

Entre descartar e preencher com outro sensor existe o cenário ideal de
**compor com outra data do próprio CBERS**, se você tiver uma cena
próxima e limpa naquele pedaço. Fica na mesma radiometria e na mesma
resolução, sem super-resolução nenhuma.

In [12]:
n_validos = int((nativa != NODATA_MASCARA).sum())
n_nuvem = int(np.isin(nativa, [1, 2, 3]).sum())
resumo = {
    "gerado_em": datetime.now().isoformat(timespec="seconds"),
    "aula": "02", "cena": CENA_ID, "nivel": NIVEL,
    "entrada": str(entrada), "detect_res_m": DETECT_RES,
    "patch_size": PATCH, "patch_overlap": OVERLAP,
    "bandas_usadas": {"red": BANDA_RED, "green": BANDA_GREEN, "nir": BANDA_NIR},
    "px_validos": n_validos,
    "px_nuvem_sombra": n_nuvem,
    "frac_contaminada": round(n_nuvem / max(n_validos, 1), 4),
    "area_contaminada_ha": round(n_nuvem * nativo["res"] ** 2 / 10_000, 1),
    "por_classe": {nome: int((nativa == c).sum()) for c, nome in CLASSES.items()},
    "saidas": {"mascara_deteccao": str(cam_det), "mascara_nativa": str(cam_nat),
               "mascara_clear": str(cam_clear), "imagem_mascarada": str(mascarada)},
}
cam_nat.with_suffix(".params.json").write_text(
    json.dumps(resumo, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"contaminado por nuvem/sombra: {resumo['frac_contaminada']:.2%} da cena "
      f"({resumo['area_contaminada_ha']:,.0f} ha)")
print(f"Params: {cam_nat.with_suffix('.params.json').name}")

contaminado por nuvem/sombra: 11.26% da cena (2,026 ha)
Params: pansharpening_20260301_boa_cloudmask_native.params.json

## O que ficou pronto

    data/interim/pansharpening_<DATA>_<nivel>_cloudmask_10m.tif      máscara na grade de detecção
    data/interim/pansharpening_<DATA>_<nivel>_cloudmask_native.tif   máscara em 2 m   <- Aula 03
    data/interim/pansharpening_<DATA>_<nivel>_clearmask_native.tif   binária, 1 = utilizável
    data/interim/pansharpening_<DATA>_<nivel>_masked.tif             cena com lacunas <- Aula 03

Na **Aula 03** essas lacunas são preenchidas com Sentinel-2 levado de 10
m para 2,5 m pelo SEN2SR, coregistrado com o CBERS pelo AROSICS e
harmonizado radiometricamente antes da emenda.

## Referências

- Wright, N. et al. (2025). *Training sensor-agnostic deep learning
  models for remote sensing: Achieving state-of-the-art cloud and cloud
  shadow identification with OmniCloudMask*. Remote Sensing of
  Environment.
- [OmniCloudMask no GitHub](https://github.com/DPIRD-DMA/OmniCloudMask)

WRIGHT, Nicholas *et al.* [Training sensor-agnostic deep learning models
for remote sensing: Achieving state-of-the-art cloud and cloud shadow
identification with
OmniCloudMask](https://doi.org/10.1016/j.rse.2025.114694). **Remote
Sensing of Environment**, v. 322, p. 114694, 2025.